# SQL Basics — AI Engineering Interview Prep

All queries run in-memory using DuckDB. No external database needed.

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect()  # in-memory

# Create sample tables
conn.execute("""
CREATE TABLE departments AS
SELECT * FROM (VALUES
  (1, 'Engineering', 'San Francisco'),
  (2, 'Marketing',   'New York'),
  (3, 'Sales',       'Chicago'),
  (4, 'HR',          'Austin'),
  (5, 'Finance',     'New York')
) t(dept_id, dept_name, location)
""")

conn.execute("""
CREATE TABLE employees AS
SELECT * FROM (VALUES
  (1, 'Alice',   1, 95000.0, '2020-01-15', 3),
  (2, 'Bob',     2, 75000.0, '2019-06-01', 2),
  (3, 'Charlie', 1, 85000.0, '2021-03-20', 1),
  (4, 'Diana',   3, 70000.0, '2018-11-05', 2),
  (5, 'Edward',  1, 105000.0,'2017-08-10', 1),
  (6, 'Fiona',   2, 82000.0, '2022-01-01', 3),
  (7, 'George',  4, 68000.0, '2020-07-15', NULL),
  (8, 'Hannah',  3, 91000.0, '2019-09-30', 4),
  (9, 'Ivan',    5, 88000.0, '2021-12-01', 3),
  (10,'Jane',    1, 115000.0,'2016-04-01', NULL)
) t(emp_id, name, dept_id, salary, hire_date, manager_id)
""")

print("Tables created. Employees:")
conn.execute("SELECT * FROM employees").df()

## 1. SELECT, WHERE, ORDER BY, LIMIT

In [ ]:
# Basic SELECT
print("=== Employees in Engineering (dept_id=1), salary>90k ===")
conn.execute("""
SELECT name, salary, hire_date
FROM employees
WHERE dept_id = 1
  AND salary > 90000
ORDER BY salary DESC
""").df()

In [ ]:
# BETWEEN, IN, LIKE, IS NULL
print("Salary between 70k-90k:")
print(conn.execute("SELECT name, salary FROM employees WHERE salary BETWEEN 70000 AND 90000").df())

print("\nIn Engineering or Finance:")
print(conn.execute("SELECT name, dept_id FROM employees WHERE dept_id IN (1, 5)").df())

print("\nNames starting with 'A' or 'B':")
print(conn.execute("SELECT name FROM employees WHERE name LIKE 'A%' OR name LIKE 'B%'").df())

print("\nNo manager (NULL):")
print(conn.execute("SELECT name FROM employees WHERE manager_id IS NULL").df())

## 2. GROUP BY, HAVING, DISTINCT

In [ ]:
print("=== Department stats ===")
conn.execute("""
SELECT
    dept_id,
    COUNT(*)          AS headcount,
    ROUND(AVG(salary), 0) AS avg_salary,
    MIN(salary)       AS min_salary,
    MAX(salary)       AS max_salary,
    SUM(salary)       AS total_payroll
FROM employees
GROUP BY dept_id
HAVING COUNT(*) > 1
ORDER BY avg_salary DESC
""").df()

## 3. String & Date Functions

In [ ]:
conn.execute("""
SELECT
    name,
    UPPER(name)        AS name_upper,
    LENGTH(name)       AS name_len,
    SUBSTRING(name, 1, 3) AS name_prefix,
    REPLACE(name, 'a', '@') AS name_modified,
    EXTRACT(year FROM hire_date::DATE)  AS hire_year,
    EXTRACT(month FROM hire_date::DATE) AS hire_month,
    DATE_DIFF('year', hire_date::DATE, CURRENT_DATE) AS years_employed
FROM employees
ORDER BY hire_date
LIMIT 5
""").df()

## 4. CASE WHEN

In [ ]:
conn.execute("""
SELECT
    name,
    salary,
    CASE
        WHEN salary >= 100000 THEN 'Senior'
        WHEN salary >= 80000  THEN 'Mid'
        ELSE 'Junior'
    END AS level,
    CASE WHEN manager_id IS NULL THEN 'IC' ELSE 'Reports to ' || CAST(manager_id AS VARCHAR) END AS role
FROM employees
ORDER BY salary DESC
""").df()

## Interview Tips

- `WHERE` filters **before** GROUP BY; `HAVING` filters **after** aggregation.
- `DISTINCT` removes duplicate rows; often combined with COUNT: `COUNT(DISTINCT col)`.
- `IS NULL` / `IS NOT NULL` — never use `= NULL` (always evaluates to NULL, not TRUE).
- Order of SQL clauses: SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT.
- `BETWEEN a AND b` is inclusive on both ends.